In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/indian_telecom_churn.csv")
df.shape

(10000, 20)

In [2]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print(categorical_cols)

for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].unique()[:10])

['customer_id', 'name', 'gender', 'circle', 'circle_type', 'operator', 'plan_type', 'payment_method', 'registration_date']

customer_id: 10000 unique values
<StringArray>
['CUST100000', 'CUST100001', 'CUST100002', 'CUST100003', 'CUST100004',
 'CUST100005', 'CUST100006', 'CUST100007', 'CUST100008', 'CUST100009']
Length: 10, dtype: str

name: 9814 unique values
<StringArray>
[   'Aryan Maharaj',      'Udant Dewan',       'Gagan Sami',
 'Ayushman Chander',     'Viraj Tiwari',     'Rushil Saini',
      'Saumya Mall',   'Nathaniel Sami',    'Arunima Dugal',
 'Gayathri Chaudry']
Length: 10, dtype: str

gender: 2 unique values
<StringArray>
['Male', 'Female']
Length: 2, dtype: str

circle: 17 unique values
<StringArray>
[       'Tamil Nadu',         'Karnataka',            'Mumbai',
           'UP East',           'Gujarat',    'Madhya Pradesh',
             'Delhi',            'Punjab',       'West Bengal',
 'Bihar & Jharkhand']
Length: 10, dtype: str

circle_type: 3 unique values
<StringArr

C:\Users\SUN\AppData\Local\Temp\ipykernel_27860\4206580066.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns.tolist()


In [3]:
# customer_id, name, registration_date won't help predict churn directly
df_model = df.drop(columns=["customer_id", "name", "registration_date"])
df_model.shape


(10000, 17)

In [4]:
cols_to_encode = ["gender", "circle", "circle_type", "operator", "plan_type", "payment_method"]

df_encoded = pd.get_dummies(df_model, columns=cols_to_encode, drop_first=True)
print(df_encoded.shape)
df_encoded.head()

(10000, 38)


,age,plan_amount_inr,tenure_months,data_usage_gb,days_since_last_recharge,num_complaints_6m,network_rating,has_bundle,uses_5g,is_active_vlr,...,circle_type_Rural,circle_type_Urban,operator_BSNL,operator_Jio,operator_Vi,plan_type_Prepaid,payment_method_Debit/Credit Card,payment_method_Net Banking,payment_method_UPI,payment_method_Wallet (Paytm/PhonePe)
0,24,239,15,15.8,6,1,3,1,1,1,...,False,True,False,True,False,True,False,False,False,False
1,41,599,31,1.4,11,0,3,0,0,0,...,False,True,True,False,False,False,True,False,False,False
2,49,199,116,2.6,41,0,5,0,0,1,...,False,False,False,False,False,True,False,False,False,False
3,19,199,1,7.9,5,1,4,0,0,1,...,True,False,False,False,False,True,True,False,False,False
4,54,149,6,5.3,1,1,3,0,0,1,...,False,True,False,True,False,True,False,False,False,False


In [5]:
print(df_encoded.columns.tolist())

['age', 'plan_amount_inr', 'tenure_months', 'data_usage_gb', 'days_since_last_recharge', 'num_complaints_6m', 'network_rating', 'has_bundle', 'uses_5g', 'is_active_vlr', 'churn', 'gender_Male', 'circle_Assam & North East', 'circle_Bihar & Jharkhand', 'circle_Delhi', 'circle_Gujarat', 'circle_Karnataka', 'circle_Kerala', 'circle_Madhya Pradesh', 'circle_Maharashtra', 'circle_Mumbai', 'circle_Odisha', 'circle_Punjab', 'circle_Rajasthan', 'circle_Tamil Nadu', 'circle_UP East', 'circle_UP West', 'circle_West Bengal', 'circle_type_Rural', 'circle_type_Urban', 'operator_BSNL', 'operator_Jio', 'operator_Vi', 'plan_type_Prepaid', 'payment_method_Debit/Credit Card', 'payment_method_Net Banking', 'payment_method_UPI', 'payment_method_Wallet (Paytm/PhonePe)']


In [6]:
df_encoded["churn"].value_counts()
df_encoded.dtypes.value_counts()

bool       27
int64      10
float64     1
Name: count, dtype: int64

In [7]:
# 1. High complaint flag (binary) - based on EDA finding that 2+ complaints show elevated churn
df_encoded["high_complaints_flag"] = (df["num_complaints_6m"] >= 2).astype(int)

# 2. Dormant customer flag - based on EDA finding of 45+ day threshold effect
df_encoded["dormant_45d_flag"] = (df["days_since_last_recharge"] > 45).astype(int)

# 3. Tenure bucket (categorical -> one-hot) - captures non-linear tenure effects
tenure_bucket = pd.cut(
    df["tenure_months"],
    bins=[-1, 3, 12, 36, 200],
    labels=["new_0_3m", "growing_4_12m", "established_13_36m", "loyal_36m_plus"]
)
tenure_dummies = pd.get_dummies(tenure_bucket, prefix="tenure", drop_first=True)
df_encoded = pd.concat([df_encoded, tenure_dummies], axis=1)

print(df_encoded.shape)
df_encoded[["high_complaints_flag", "dormant_45d_flag"]].sum()

(10000, 43)


high_complaints_flag    1158
dormant_45d_flag        1000
dtype: int64

In [8]:
print(df_encoded.groupby("high_complaints_flag")["churn"].mean())
print(df_encoded.groupby("dormant_45d_flag")["churn"].mean())

high_complaints_flag
0    0.093305
1    0.166667
Name: churn, dtype: float64
dormant_45d_flag
0    0.089556
1    0.212000
Name: churn, dtype: float64


In [9]:
X = df_encoded.drop(columns=["churn"])
y = df_encoded["churn"]

print(X.shape, y.shape)
print(y.value_counts(normalize=True))

(10000, 42) (10000,)
churn
0    0.8982
1    0.1018
Name: proportion, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())

Train shape: (8000, 42)
Test shape: (2000, 42)

Train churn rate: 0.10175
Test churn rate: 0.102


In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved processed datasets")

Saved processed datasets


## Phase 2 Summary: Preprocessing

1. Dropped identifier columns (customer_id, name, registration_date) - no predictive value.
2. One-hot encoded 6 nominal categorical variables (gender, circle, circle_type, operator, 
   plan_type, payment_method) with drop_first=True to avoid multicollinearity.
3. Engineered 2 binary flags (high_complaints_flag, dormant_45d_flag) and tenure buckets 
   directly from threshold effects discovered in EDA.
4. Final feature set: 42 columns, 10,000 rows.
5. Train/test split (80/20) with stratification - churn rate preserved at ~10.2% in both sets.
6. Class imbalance (10% churn) will be handled via class_weight="balanced" in Phase 3 models,
   chosen over SMOTE to avoid data leakage risk.

**Next**: Phase 3 - churn prediction modeling (Logistic Regression, Random Forest/XGBoost, 
SHAP interpretation).